# Resumo das Estruturas de Armazenamento em Árvores

Este notebook implementa e demonstra didaticamente as seguintes estruturas: ABB, AVL, Trie, Rubro-Negra, 2-3, B e B+.

Para cada estrutura, o notebook contém:
- Célula de implementação com comentários e complexidades de inserir, buscar e remover.
- Célula de simulação de testes para inserir, pesquisar e remover elementos.
- Célula separada de visualização (HTML/SVG) da estrutura montada.

As visualizações foram organizadas para facilitar a observação da forma da árvore após operações típicas.

## Estrutura simplificada das células
1. Resumo geral e estrutura do notebook
2. Utilitários de visualização (SVG/HTML)
3. ABB - implementação
4. ABB - simulação de testes
5. ABB - visualização
6. AVL - implementação
7. AVL - simulação de testes
8. AVL - visualização
9. Trie - implementação
10. Trie - simulação de testes
11. Trie - visualização
12. Rubro-Negra - implementação
13. Rubro-Negra - simulação de testes
14. Rubro-Negra - visualização
15. 2-3 - implementação
16. 2-3 - simulação de testes
17. 2-3 - visualização
18. B - implementação
19. B - simulação de testes
20. B - visualização
21. B+ - implementação
22. B+ - simulação de testes
23. B+ - visualização

In [1]:
# Utilitários comuns de visualização em HTML/SVG para árvores do notebook.
# Complexidade dos utilitários (layout e renderização): O(n), onde n é o número de nós exibidos.
from collections import deque
from dataclasses import dataclass, field
from html import escape
from IPython.display import HTML, display

def render_tree_html(root, get_children, label_fn=str, title='Árvore', edge_label_fn=None):
    if root is None:
        display(HTML(f"<h4>{escape(title)}</h4><p><i>Árvore vazia.</i></p>"))
        return

    q = deque([(root, 0)])
    levels = {}
    parent_of = {}
    edge_labels = {}
    seen = set()

    while q:
        node, lvl = q.popleft()
        node_id = id(node)
        if node_id in seen:
            continue
        seen.add(node_id)
        levels.setdefault(lvl, []).append(node)

        for child in [c for c in get_children(node) if c is not None]:
            c_id = id(child)
            if c_id not in parent_of:
                parent_of[c_id] = node
                if edge_label_fn is not None:
                    edge_labels[(node_id, c_id)] = edge_label_fn(node, child)
            q.append((child, lvl + 1))

    level_count = max(levels.keys()) + 1
    width = max(900, max(len(v) for v in levels.values()) * 140)
    height = 120 + level_count * 120
    positions = {}

    for lvl in range(level_count):
        nodes = levels.get(lvl, [])
        if not nodes:
            continue
        step = width / (len(nodes) + 1)
        y = 70 + lvl * 110
        for idx, node in enumerate(nodes, start=1):
            positions[id(node)] = (idx * step, y)

    svg_parts = [f"<h4>{escape(title)}</h4>", f"<svg width='{width}' height='{height}' style='background:#f8fafc;border:1px solid #d1d5db;border-radius:8px;'>"]

    for child_id, parent in parent_of.items():
        parent_id = id(parent)
        if parent_id in positions and child_id in positions:
            x1, y1 = positions[parent_id]
            x2, y2 = positions[child_id]
            svg_parts.append(f"<line x1='{x1}' y1='{y1+20}' x2='{x2}' y2='{y2-20}' stroke='#64748b' stroke-width='2'/>")
            e = edge_labels.get((parent_id, child_id), '')
            if e not in (None, ''):
                mx, my = (x1 + x2) / 2, (y1 + y2) / 2
                svg_parts.append(f"<text x='{mx}' y='{my-6}' text-anchor='middle' font-size='12' fill='#334155'>{escape(str(e))}</text>")

    for node_id, (x, y) in positions.items():
        node = next(n for lv in levels.values() for n in lv if id(n) == node_id)
        txt = escape(str(label_fn(node)))
        svg_parts.append(f"<circle cx='{x}' cy='{y}' r='22' fill='#0ea5e9' stroke='#075985' stroke-width='2'/>")
        svg_parts.append(f"<text x='{x}' y='{y+5}' text-anchor='middle' fill='white' font-size='12' font-family='monospace'>{txt}</text>")

    svg_parts.append('</svg>')
    display(HTML(''.join(svg_parts)))

In [2]:
# implementação da ABB (Árvore Binária de Busca).
# Complexidades (caso médio): inserir O(log n), buscar O(log n), remover O(log n).
# Pior caso (árvore degenerada): inserir/buscar/remover O(n).
@dataclass
class ABBNode:
    key: int
    left: 'ABBNode | None' = None
    right: 'ABBNode | None' = None

class ABB:
    def __init__(self):
        self.root = None

    def insert(self, key):
        def _insert(node, key):
            if node is None:
                return ABBNode(key)
            if key < node.key:
                node.left = _insert(node.left, key)
            elif key > node.key:
                node.right = _insert(node.right, key)
            return node
        self.root = _insert(self.root, key)

    def search(self, key):
        node = self.root
        while node is not None:
            if key == node.key:
                return True
            node = node.left if key < node.key else node.right
        return False

    def remove(self, key):
        def _min_node(node):
            while node.left is not None:
                node = node.left
            return node

        def _remove(node, key):
            if node is None:
                return None
            if key < node.key:
                node.left = _remove(node.left, key)
            elif key > node.key:
                node.right = _remove(node.right, key)
            else:
                if node.left is None:
                    return node.right
                if node.right is None:
                    return node.left
                succ = _min_node(node.right)
                node.key = succ.key
                node.right = _remove(node.right, succ.key)
            return node

        self.root = _remove(self.root, key)

    def inorder(self):
        out = []
        def _dfs(node):
            if node is None:
                return
            _dfs(node.left)
            out.append(node.key)
            _dfs(node.right)
        _dfs(self.root)
        return out

In [3]:
# simulação de testes da ABB para inserir, pesquisar e remover elementos.
abb = ABB()
dados_abb = [40, 20, 60, 10, 30, 50, 70, 25]
for x in dados_abb:
    abb.insert(x)

print('ABB após inserções (inorder):', abb.inorder())
print('Buscar 30:', abb.search(30))
print('Buscar 99:', abb.search(99))

abb.remove(20)
abb.remove(70)
print('ABB após remoções 20 e 70 (inorder):', abb.inorder())

ABB após inserções (inorder): [10, 20, 25, 30, 40, 50, 60, 70]
Buscar 30: True
Buscar 99: False
ABB após remoções 20 e 70 (inorder): [10, 25, 30, 40, 50, 60]


In [4]:
# visualização didática da ABB montada em SVG/HTML (separada da implementação e testes).
render_tree_html(
    abb.root,
    get_children=lambda n: [n.left, n.right],
    label_fn=lambda n: n.key,
    title='Visualização da ABB'
)

In [5]:
# implementação da AVL (ABB auto-balanceada por rotações).
# Complexidades: inserir O(log n), buscar O(log n), remover O(log n).
@dataclass
class AVLNode:
    key: int
    left: 'AVLNode | None' = None
    right: 'AVLNode | None' = None
    height: int = 1

class AVL:
    def __init__(self):
        self.root = None

    def _h(self, node):
        return node.height if node else 0

    def _bf(self, node):
        return self._h(node.left) - self._h(node.right) if node else 0

    def _upd(self, node):
        node.height = 1 + max(self._h(node.left), self._h(node.right))

    def _rot_r(self, y):
        x = y.left
        t2 = x.right
        x.right = y
        y.left = t2
        self._upd(y)
        self._upd(x)
        return x

    def _rot_l(self, x):
        y = x.right
        t2 = y.left
        y.left = x
        x.right = t2
        self._upd(x)
        self._upd(y)
        return y

    def _balance(self, node):
        self._upd(node)
        bf = self._bf(node)

        if bf > 1:
            if self._bf(node.left) < 0:
                node.left = self._rot_l(node.left)
            return self._rot_r(node)

        if bf < -1:
            if self._bf(node.right) > 0:
                node.right = self._rot_r(node.right)
            return self._rot_l(node)

        return node

    def insert(self, key):
        def _insert(node, key):
            if node is None:
                return AVLNode(key)
            if key < node.key:
                node.left = _insert(node.left, key)
            elif key > node.key:
                node.right = _insert(node.right, key)
            return self._balance(node)
        self.root = _insert(self.root, key)

    def search(self, key):
        node = self.root
        while node:
            if key == node.key:
                return True
            node = node.left if key < node.key else node.right
        return False

    def remove(self, key):
        def _min_node(node):
            while node.left:
                node = node.left
            return node

        def _remove(node, key):
            if node is None:
                return None
            if key < node.key:
                node.left = _remove(node.left, key)
            elif key > node.key:
                node.right = _remove(node.right, key)
            else:
                if node.left is None:
                    return node.right
                if node.right is None:
                    return node.left
                succ = _min_node(node.right)
                node.key = succ.key
                node.right = _remove(node.right, succ.key)
            return self._balance(node) if node else None

        self.root = _remove(self.root, key)

    def inorder(self):
        out = []
        def _dfs(node):
            if node is None:
                return
            _dfs(node.left)
            out.append(node.key)
            _dfs(node.right)
        _dfs(self.root)
        return out

In [6]:
# simulação de testes da AVL para inserir, pesquisar e remover elementos.
avl = AVL()
dados_avl = [30, 20, 40, 10, 25, 35, 50, 5]
for x in dados_avl:
    avl.insert(x)

print('AVL após inserções (inorder):', avl.inorder())
print('Buscar 35:', avl.search(35))
print('Buscar 99:', avl.search(99))

avl.remove(20)
avl.remove(50)
print('AVL após remoções 20 e 50 (inorder):', avl.inorder())

AVL após inserções (inorder): [5, 10, 20, 25, 30, 35, 40, 50]
Buscar 35: True
Buscar 99: False
AVL após remoções 20 e 50 (inorder): [5, 10, 25, 30, 35, 40]


In [7]:
# Visualização didática da AVL montada em SVG/HTML.
render_tree_html(
    avl.root,
    get_children=lambda n: [n.left, n.right],
    label_fn=lambda n: f"{n.key}|h{n.height}",
    title='Visualização da AVL'
)

In [8]:
# implementação da Trie (árvore de prefixos para strings).
# Complexidades: inserir O(m), buscar O(m), remover O(m), onde m é o tamanho da palavra.
@dataclass
class TrieNode:
    children: dict = field(default_factory=dict)
    end: bool = False
    ch: str = ''

class Trie:
    def __init__(self):
        self.root = TrieNode(ch='*')

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode(ch=ch)
            node = node.children[ch]
        node.end = True

    def search(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                return False
            node = node.children[ch]
        return node.end

    def remove(self, word):
        def _remove(node, i):
            if i == len(word):
                if not node.end:
                    return False, False
                node.end = False
                return True, len(node.children) == 0

            ch = word[i]
            if ch not in node.children:
                return False, False

            removed, delete_child = _remove(node.children[ch], i + 1)
            if delete_child:
                del node.children[ch]

            should_delete = (not node.end) and (len(node.children) == 0)
            return removed, should_delete

        removed, _ = _remove(self.root, 0)
        return removed

In [9]:
# simulação de testes da Trie para inserir, pesquisar e remover elementos.
trie = Trie()
palavras = ['casa', 'caso', 'cama', 'carro', 'carta']
for p in palavras:
    trie.insert(p)

print('Buscar casa:', trie.search('casa'))
print('Buscar cas:', trie.search('cas'))
print('Buscar carta:', trie.search('carta'))

trie.remove('caso')
trie.remove('cama')
print('Buscar caso após remoção:', trie.search('caso'))
print('Buscar casa (permanece):', trie.search('casa'))

Buscar casa: True
Buscar cas: False
Buscar carta: True
Buscar caso após remoção: False
Buscar casa (permanece): True


In [10]:
# Visualização didática da Trie montada em SVG/HTML, com rótulos de aresta por caractere.
def trie_edge_label(parent, child):
    return child.ch

render_tree_html(
    trie.root,
    get_children=lambda n: list(n.children.values()),
    label_fn=lambda n: f"{n.ch}{'*' if n.end else ''}",
    title='Visualização da Trie',
    edge_label_fn=trie_edge_label
)

In [11]:
# implementação de Árvore Rubro-Negra (inserção balanceada por recoloração/rotações).
# Complexidades teóricas: inserir O(log n), buscar O(log n), remover O(log n).
# Nesta versão didática, a remoção é simulada por reconstrução da árvore após excluir a chave.
@dataclass
class RBNode:
    key: int
    color: str = 'R'
    left: 'RBNode | None' = None
    right: 'RBNode | None' = None
    parent: 'RBNode | None' = None

class RBTree:
    def __init__(self):
        self.root = None

    def _rot_l(self, x):
        y = x.right
        x.right = y.left
        if y.left:
            y.left.parent = x
        y.parent = x.parent
        if x.parent is None:
            self.root = y
        elif x == x.parent.left:
            x.parent.left = y
        else:
            x.parent.right = y
        y.left = x
        x.parent = y

    def _rot_r(self, y):
        x = y.left
        y.left = x.right
        if x.right:
            x.right.parent = y
        x.parent = y.parent
        if y.parent is None:
            self.root = x
        elif y == y.parent.left:
            y.parent.left = x
        else:
            y.parent.right = x
        x.right = y
        y.parent = x

    def insert(self, key):
        z = RBNode(key=key)
        y = None
        x = self.root
        while x is not None:
            y = x
            if z.key < x.key:
                x = x.left
            elif z.key > x.key:
                x = x.right
            else:
                return
        z.parent = y
        if y is None:
            self.root = z
        elif z.key < y.key:
            y.left = z
        else:
            y.right = z
        self._fix_insert(z)

    def _fix_insert(self, z):
        while z.parent and z.parent.color == 'R':
            if z.parent == z.parent.parent.left:
                y = z.parent.parent.right
                if y and y.color == 'R':
                    z.parent.color = 'B'
                    y.color = 'B'
                    z.parent.parent.color = 'R'
                    z = z.parent.parent
                else:
                    if z == z.parent.right:
                        z = z.parent
                        self._rot_l(z)
                    z.parent.color = 'B'
                    z.parent.parent.color = 'R'
                    self._rot_r(z.parent.parent)
            else:
                y = z.parent.parent.left
                if y and y.color == 'R':
                    z.parent.color = 'B'
                    y.color = 'B'
                    z.parent.parent.color = 'R'
                    z = z.parent.parent
                else:
                    if z == z.parent.left:
                        z = z.parent
                        self._rot_r(z)
                    z.parent.color = 'B'
                    z.parent.parent.color = 'R'
                    self._rot_l(z.parent.parent)
        if self.root:
            self.root.color = 'B'

    def search(self, key):
        n = self.root
        while n:
            if key == n.key:
                return True
            n = n.left if key < n.key else n.right
        return False

    def inorder(self):
        out = []
        def _dfs(n):
            if n is None:
                return
            _dfs(n.left)
            out.append(n.key)
            _dfs(n.right)
        _dfs(self.root)
        return out

    def remove(self, key):
        keys = self.inorder()
        if key not in keys:
            return False
        self.root = None
        for k in keys:
            if k != key:
                self.insert(k)
        return True

In [12]:
# simulação de testes da Rubro-Negra para inserir, pesquisar e remover elementos.
rb = RBTree()
dados_rb = [41, 38, 31, 12, 19, 8, 50, 60]
for x in dados_rb:
    rb.insert(x)

print('RB após inserções (inorder):', rb.inorder())
print('Buscar 19:', rb.search(19))
print('Buscar 99:', rb.search(99))

rb.remove(31)
rb.remove(60)
print('RB após remoções 31 e 60 (inorder):', rb.inorder())

RB após inserções (inorder): [8, 12, 19, 31, 38, 41, 50, 60]
Buscar 19: True
Buscar 99: False
RB após remoções 31 e 60 (inorder): [8, 12, 19, 38, 41, 50]


In [13]:
# Visualização didática da Rubro-Negra em SVG/HTML, indicando chave e cor (R/B).
render_tree_html(
    rb.root,
    get_children=lambda n: [n.left, n.right],
    label_fn=lambda n: f"{n.key}{n.color}",
    title='Visualização da Rubro-Negra'
)

In [14]:
# implementação didática de Árvore 2-3 (nós com 1 ou 2 chaves; 2 ou 3 filhos).
# Complexidades teóricas: inserir O(log n), buscar O(log n), remover O(log n).
# Nesta versão, a remoção é simulada por reconstrução após excluir a chave.
@dataclass
class TTNode:
    keys: list = field(default_factory=list)
    children: list = field(default_factory=list)

    @property
    def leaf(self):
        return len(self.children) == 0

class TwoThreeTree:
    def __init__(self):
        self.root = None

    def search(self, key):
        node = self.root
        while node:
            if key in node.keys:
                return True
            if node.leaf:
                return False
            if key < node.keys[0]:
                node = node.children[0]
            elif len(node.keys) == 1 or key < node.keys[1]:
                node = node.children[1]
            else:
                node = node.children[2]
        return False

    def insert(self, key):
        if self.root is None:
            self.root = TTNode(keys=[key])
            return

        split = self._insert_rec(self.root, key)
        if split:
            mid, left, right = split
            self.root = TTNode(keys=[mid], children=[left, right])

    def _insert_rec(self, node, key):
        if node.leaf:
            if key in node.keys:
                return None
            node.keys.append(key)
            node.keys.sort()
            if len(node.keys) <= 2:
                return None
            return self._split_node(node)

        if key < node.keys[0]:
            idx = 0
        elif len(node.keys) == 1 or key < node.keys[1]:
            idx = 1
        else:
            idx = 2

        split = self._insert_rec(node.children[idx], key)
        if not split:
            return None

        mid, left, right = split
        node.keys.insert(idx, mid)
        node.children[idx] = left
        node.children.insert(idx + 1, right)

        if len(node.keys) <= 2:
            return None
        return self._split_node(node)

    def _split_node(self, node):
        mid = node.keys[1]
        left = TTNode(keys=[node.keys[0]])
        right = TTNode(keys=[node.keys[2]])
        if node.children:
            left.children = node.children[:2]
            right.children = node.children[2:]
        return mid, left, right

    def inorder(self):
        out = []
        def _walk(node):
            if node is None:
                return
            if node.leaf:
                out.extend(node.keys)
                return
            if len(node.keys) == 1:
                _walk(node.children[0])
                out.append(node.keys[0])
                _walk(node.children[1])
            else:
                _walk(node.children[0])
                out.append(node.keys[0])
                _walk(node.children[1])
                out.append(node.keys[1])
                _walk(node.children[2])
        _walk(self.root)
        return out

    def remove(self, key):
        vals = self.inorder()
        if key not in vals:
            return False
        self.root = None
        for v in vals:
            if v != key:
                self.insert(v)
        return True

In [15]:
# simulação de testes da Árvore 2-3 para inserir, pesquisar e remover elementos.
tt = TwoThreeTree()
dados_tt = [15, 5, 25, 2, 8, 20, 30, 18, 22]
for x in dados_tt:
    tt.insert(x)

print('2-3 após inserções (inorder):', tt.inorder())
print('Buscar 18:', tt.search(18))
print('Buscar 99:', tt.search(99))

tt.remove(8)
tt.remove(25)
print('2-3 após remoções 8 e 25 (inorder):', tt.inorder())

2-3 após inserções (inorder): [2, 5, 8, 15, 18, 20, 22, 25, 30]
Buscar 18: True
Buscar 99: False
2-3 após remoções 8 e 25 (inorder): [2, 5, 15, 18, 20, 22, 30]


In [16]:
# Visualização didática da Árvore 2-3 em SVG/HTML.
render_tree_html(
    tt.root,
    get_children=lambda n: n.children,
    label_fn=lambda n: '|'.join(str(k) for k in n.keys),
    title='Visualização da Árvore 2-3'
)

In [17]:
# implementação de Árvore B com grau mínimo t=3 (cada nó possui várias chaves).
# Complexidades: inserir O(log n), buscar O(log n), remover O(log n).
# Nesta versão didática, a remoção é simulada por reconstrução após excluir a chave.
@dataclass
class BNode:
    t: int
    leaf: bool = True
    keys: list = field(default_factory=list)
    children: list = field(default_factory=list)

class BTree:
    def __init__(self, t=3):
        self.t = t
        self.root = BNode(t=t, leaf=True)

    def search(self, key, node=None):
        node = self.root if node is None else node
        i = 0
        while i < len(node.keys) and key > node.keys[i]:
            i += 1
        if i < len(node.keys) and key == node.keys[i]:
            return True
        if node.leaf:
            return False
        return self.search(key, node.children[i])

    def split_child(self, parent, i):
        t = self.t
        y = parent.children[i]
        z = BNode(t=t, leaf=y.leaf)

        mid = y.keys[t - 1]
        z.keys = y.keys[t:]
        y.keys = y.keys[:t - 1]

        if not y.leaf:
            z.children = y.children[t:]
            y.children = y.children[:t]

        parent.children.insert(i + 1, z)
        parent.keys.insert(i, mid)

    def insert(self, key):
        r = self.root
        if len(r.keys) == 2 * self.t - 1:
            s = BNode(t=self.t, leaf=False, children=[r])
            self.root = s
            self.split_child(s, 0)
            self._insert_non_full(s, key)
        else:
            self._insert_non_full(r, key)

    def _insert_non_full(self, node, key):
        i = len(node.keys) - 1
        if node.leaf:
            node.keys.append(0)
            while i >= 0 and key < node.keys[i]:
                node.keys[i + 1] = node.keys[i]
                i -= 1
            if i >= 0 and node.keys[i] == key:
                node.keys.pop()
                return
            node.keys[i + 1] = key
        else:
            while i >= 0 and key < node.keys[i]:
                i -= 1
            i += 1
            if len(node.children[i].keys) == 2 * self.t - 1:
                self.split_child(node, i)
                if key > node.keys[i]:
                    i += 1
            self._insert_non_full(node.children[i], key)

    def traverse(self):
        out = []
        def _walk(node):
            if node.leaf:
                out.extend(node.keys)
                return
            for i, k in enumerate(node.keys):
                _walk(node.children[i])
                out.append(k)
            _walk(node.children[-1])
        _walk(self.root)
        return out

    def remove(self, key):
        vals = self.traverse()
        if key not in vals:
            return False
        self.root = BNode(t=self.t, leaf=True)
        for v in vals:
            if v != key:
                self.insert(v)
        return True

In [18]:
# simulação de testes da Árvore B para inserir, pesquisar e remover elementos.
bt = BTree(t=3)
dados_bt = [10, 20, 5, 6, 12, 30, 7, 17, 3, 25, 40]
for x in dados_bt:
    bt.insert(x)

print('B após inserções (ordem):', bt.traverse())
print('Buscar 17:', bt.search(17))
print('Buscar 99:', bt.search(99))

bt.remove(6)
bt.remove(30)
print('B após remoções 6 e 30 (ordem):', bt.traverse())

B após inserções (ordem): [3, 5, 6, 7, 10, 12, 17, 20, 25, 30, 40]
Buscar 17: True
Buscar 99: False
B após remoções 6 e 30 (ordem): [3, 5, 7, 10, 12, 17, 20, 25, 40]


In [19]:
# Visualização didática da Árvore B em SVG/HTML.
render_tree_html(
    bt.root,
    get_children=lambda n: n.children,
    label_fn=lambda n: '[' + ','.join(str(k) for k in n.keys) + ']',
    title='Visualização da Árvore B (t=3)'
)

In [20]:
# implementação de Árvore B+ (dados nas folhas e folhas encadeadas).
# Complexidades: inserir O(log n), buscar O(log n), remover O(log n).
# Nesta versão didática, a remoção é simulada por reconstrução após excluir a chave.
@dataclass
class BPlusNode:
    leaf: bool = True
    keys: list = field(default_factory=list)
    children: list = field(default_factory=list)
    next: 'BPlusNode | None' = None

class BPlusTree:
    def __init__(self, order=4):
        self.order = order
        self.root = BPlusNode(leaf=True)

    def search(self, key):
        node = self.root
        while not node.leaf:
            i = 0
            while i < len(node.keys) and key >= node.keys[i]:
                i += 1
            node = node.children[i]
        return key in node.keys

    def insert(self, key):
        path = []
        node = self.root
        while not node.leaf:
            path.append(node)
            i = 0
            while i < len(node.keys) and key >= node.keys[i]:
                i += 1
            node = node.children[i]

        if key in node.keys:
            return

        node.keys.append(key)
        node.keys.sort()

        if len(node.keys) < self.order:
            return

        self._split_leaf(node, path)

    def _split_leaf(self, leaf, path):
        mid = len(leaf.keys) // 2
        right = BPlusNode(leaf=True)
        right.keys = leaf.keys[mid:]
        leaf.keys = leaf.keys[:mid]
        right.next = leaf.next
        leaf.next = right
        promoted = right.keys[0]
        self._insert_in_parent(leaf, promoted, right, path)

    def _insert_in_parent(self, left, key, right, path):
        if not path:
            new_root = BPlusNode(leaf=False, keys=[key], children=[left, right])
            self.root = new_root
            return

        parent = path.pop()
        i = parent.children.index(left)
        parent.keys.insert(i, key)
        parent.children.insert(i + 1, right)

        if len(parent.keys) < self.order:
            return

        self._split_internal(parent, path)

    def _split_internal(self, node, path):
        mid = len(node.keys) // 2
        promoted = node.keys[mid]

        right = BPlusNode(leaf=False)
        right.keys = node.keys[mid + 1:]
        right.children = node.children[mid + 1:]

        node.keys = node.keys[:mid]
        node.children = node.children[:mid + 1]

        self._insert_in_parent(node, promoted, right, path)

    def all_keys(self):
        node = self.root
        while not node.leaf:
            node = node.children[0]
        out = []
        while node:
            out.extend(node.keys)
            node = node.next
        return out

    def remove(self, key):
        vals = self.all_keys()
        if key not in vals:
            return False
        self.root = BPlusNode(leaf=True)
        for v in vals:
            if v != key:
                self.insert(v)
        return True

In [21]:
# simulação de testes da Árvore B+ para inserir, pesquisar e remover elementos.
bpt = BPlusTree(order=4)
dados_bpt = [11, 3, 7, 21, 15, 30, 18, 25, 1, 5]
for x in dados_bpt:
    bpt.insert(x)

print('B+ após inserções (folhas encadeadas):', bpt.all_keys())
print('Buscar 18:', bpt.search(18))
print('Buscar 99:', bpt.search(99))

bpt.remove(7)
bpt.remove(21)
print('B+ após remoções 7 e 21 (folhas encadeadas):', bpt.all_keys())

B+ após inserções (folhas encadeadas): [1, 3, 5, 7, 11, 15, 18, 21, 25, 30]
Buscar 18: True
Buscar 99: False
B+ após remoções 7 e 21 (folhas encadeadas): [1, 3, 5, 11, 15, 18, 25, 30]


In [22]:
# Visualização didática da Árvore B+ em SVG/HTML.
render_tree_html(
    bpt.root,
    get_children=lambda n: n.children,
    label_fn=lambda n: '<' + '|'.join(str(k) for k in n.keys) + '>' if n.leaf else '[' + '|'.join(str(k) for k in n.keys) + ']',
    title='Visualização da Árvore B+'
)